# ⚡ Predição da Progressão do Diabetes com XGBoost Regressor

## 🩺 Sobre o Dataset

Este projeto utiliza o **Diabetes Dataset** da biblioteca `scikit-learn`, com informações de **442 pacientes** e **10 variáveis preditoras**, para estimar um **indicador quantitativo de progressão da doença após um ano** por meio do algoritmo **XGBoost Regressor (`XGBRegressor`)**.

- **Variável-alvo (`target`)**:
  - Número **contínuo**, entre **25 e 346** nas observações disponíveis.
  - Valores maiores representam um valor maior do indicador de progressão estudado; **não representam probabilidade, classificação de risco ou diagnóstico clínico**.
- **Variáveis utilizadas (`X`)**:
  - `age`: idade;
  - `sex`: sexo codificado;
  - `bmi`: índice de massa corporal;
  - `bp`: medida de pressão arterial;
  - `s1` a `s6`: seis medidas sanguíneas.
- **Dimensões:** 442 observações × 10 variáveis; 11 colunas incluindo o alvo.
- **Pré-processamento existente:** por padrão, o `load_diabetes()` já fornece as 10 variáveis numéricas transformadas/escalonadas. Portanto, valores como `bmi = 0.061696` **não são IMC em unidades clínicas originais**.
- **Valores ausentes:** nenhum, conforme o log de EDA.

> ⚠️ **Uso educacional:** este projeto serve para praticar regressão e avaliação de modelos. Não deve ser utilizado para diagnosticar, prever prognóstico individual ou tomar decisões clínicas.

---

## 🚀 Pipeline do Projeto

| Etapa | Descrição |
|---|---|
| 📥 **1. Carregamento** | Importação do `load_diabetes(as_frame=True)` |
| 📊 **2. Análise Exploratória** | `.head()`, `.info()`, `.describe()`, valores ausentes e correlações |
| 🔀 **3. Split dos dados** | Treino (80%) / Teste (20%), com `random_state=42` |
| 🔍 **4. Cross-validation** | `KFold` com cinco folds embaralhados, somente nos dados de treinamento |
| ⚙️ **5. GridSearchCV** | Otimização de `n_estimators`, `learning_rate`, `reg_alpha` e `reg_lambda` |
| 🤖 **6. Treinamento** | `XGBRegressor` com árvores pequenas, amostragem e regularização L1/L2 |
| 📈 **7. Avaliação** | RMSE, MAE, R², R² CV, comparação treino/teste e `DummyRegressor` como baseline |
| 📷 **8. Visualizações** | Dispersão, resíduos, real × predito, métricas, importância e variação entre folds |
| 💾 **9. Exportação** | Gráficos PNG, métricas CSV, previsões CSV, relatório JSON e modelo treinado |

**Fluxo resumido:**

`Dataset → EDA → Treino/Teste → GridSearch + KFold → Melhor XGBRegressor → Predições → Avaliação → Gráficos → Exportação`

---

## 🧠 O que é XGBoost Regressor?

**XGBoost** significa *Extreme Gradient Boosting*. É uma implementação de **Gradient Boosting baseada em árvores de decisão**. Seu funcionamento é diferente do Random Forest e do AdaBoost clássico:

- **Random Forest:** constrói árvores diversificadas, que podem ser treinadas de maneira independente, e agrega suas previsões.
- **AdaBoost clássico:** atualiza a importância das amostras ou combinações de classificadores conforme os erros de etapas anteriores.
- **XGBoost Regressor:** adiciona **árvores sequencialmente** para minimizar uma **função de perda**, utilizando informações de gradiente e curvatura (Hessiana) e recursos de regularização.

Em nosso projeto, `objective="reg:squarederror"` define a tarefa como **regressão por erro quadrático**. Por isso utilizamos **`XGBRegressor`**, não `XGBClassifier`.

### 🔁 Como o aprendizado acontece?

1. O XGBoost inicia com uma previsão de referência.
2. Calcula, para cada observação, como a função de perda varia em relação às previsões atuais.
3. Constrói uma nova árvore para aprender uma contribuição útil à redução dessa perda.
4. Reduz o tamanho da atualização utilizando o **`learning_rate`**.
5. Adiciona a contribuição ao conjunto e repete o processo conforme **`n_estimators`**.
6. Controla a complexidade usando profundidade, condições de divisão, amostragem e regularização.

No **Gradient Boosting de regressão**, as árvores não são classificadores que votam. O conjunto produz **valores numéricos contínuos**.

---

## ⚙️ Hiperparâmetros e Regularização

| Hiperparâmetro | Valores testados ou fixados | Para que serve |
|---|---|---|
| `n_estimators` | 100, 200, 300, 400 | Quantidade de árvores adicionadas sequencialmente |
| `learning_rate` | 0,01; 0,03; 0,05 | Intensidade da contribuição de cada nova árvore |
| `max_depth` | 2 (fixo) | Profundidade máxima de cada árvore |
| `min_child_weight` | 5 (fixo) | Soma mínima de Hessianas exigida em um nó filho; limita algumas divisões |
| `subsample` | 0,8 (fixo) | Fração de linhas amostradas por etapa |
| `reg_alpha` | 0; 0,1; 1 | Penalização **L1** dos pesos das folhas |
| `reg_lambda` | 1; 5; 10 | Penalização **L2** dos pesos das folhas |
| `tree_method` | `hist` | Algoritmo de construção de árvores baseado em histogramas |
| `objective` | `reg:squarederror` | Minimização de erros quadráticos |
| `random_state` | 42 | Ajuda a reproduzir os resultados |
| `n_jobs` | 1 | Reduz problemas de execução paralela no Windows/Jupyter |

### 🛡️ L1 e L2: por que utilizar?

- **L1 (`reg_alpha`)** penaliza a magnitude dos pesos das folhas, podendo zerar algumas contribuições.
- **L2 (`reg_lambda`)** penaliza valores grandes dos pesos das folhas, reduzindo sua amplitude.
- **Atenção:** são regularizações das **saídas das folhas**, e não exatamente a mesma operação aplicada aos coeficientes da regressão linear.

**Exemplo de raciocínio:** se o modelo estiver se ajustando excessivamente às observações de treino, aumentar a regularização pode reduzir a complexidade efetiva das contribuições. Entretanto, regularização excessiva também pode provocar *underfitting*.

### 🔍 Por que usar GridSearchCV?

O `GridSearchCV` testa combinações definidas pelo pesquisador e seleciona aquela com maior **R² médio na validação cruzada**. Ele **não descobre obrigatoriamente a melhor configuração possível** fora da grade.

- **Busca atual no projeto corrigido:** `4 × 3 × 3 × 3 = 108` combinações.
- **Validação:** cinco folds → **540 ajustes**, mais o refit da melhor configuração.
- **Modo `--quick`:** grade reduzida para verificar a execução sem processar a busca inteira.
- **Conjunto de teste:** permanece fora da seleção de hiperparâmetros na execução do script.

---

## 📊 Régua Visual de Avaliação de Métricas de Regressão

A régua abaixo mantém o **mesmo estilo visual do projeto de AdaBoost**. Os cortes numéricos são **faixas didáticas de referência para este exercício**, e não limites estatísticos universais nem critérios clínicos. A interpretação real depende da escala do alvo, da comparação com o baseline e da estabilidade em validação.

| Métrica | O que mede | Quando usar | Interpretação — Régua visual didática |
|---|---|---|---|
| **RMSE** | Raiz do Erro Quadrático Médio | Quando erros grandes devem pesar mais | ✅ **< 50:** menor erro relativo à régua · 🟡 **50–70:** erro intermediário · 🔴 **> 70:** erro elevado |
| **MAE** | Média dos Erros Absolutos | Para entender o tamanho médio do erro nas unidades do indicador | ✅ **< 45:** erro menor · 🟡 **45–65:** intermediário · 🔴 **> 65:** elevado |
| **R²** | Redução relativa da soma dos erros quadráticos em comparação à previsão constante pela média do conjunto avaliado | Qualidade geral do ajuste | ✅ **≥ 0,80:** ajuste muito forte · 🟡 **0,30–0,79:** ajuste parcial/moderado · 🔴 **< 0,30:** ajuste limitado pela régua |
| **R² CV** | Média dos R² obtidos nos folds de validação | Selecionar hiperparâmetros e estimar generalização | ✅ **Positivo e estável:** sinal favorável · 🟡 **Variação alta:** investigar · 🔴 **Muito abaixo de zero:** modelo inferior ao referencial dos folds |
| **Gap de R²** | Diferença entre R² de treino e teste | Investigar possível sobreajuste | ✅ **Pequeno e estável:** favorável · 🟡 **Moderado:** investigar · 🔴 **Muito elevado:** indício forte de overfitting |


### 🎯 Régua visual aplicada aos resultados reais da execução completa

| Indicador | Valor obtido | Leitura na régua | Interpretação contextualizada |
|---|---:|---|---|
| **RMSE — Treino** | **44,10** | ✅ Faixa menor | Esperado que treino tenha erro menor; não prova generalização |
| **RMSE — Teste** | **51,64** | 🟡 Faixa intermediária | **Melhor** que o baseline de 73,22 |
| **MAE — Teste** | **41,38** | ✅ Faixa menor | Erro absoluto médio de 41,38 unidades do indicador |
| **R² — Treino** | **0,6800** | 🟡 Ajuste parcial | Melhor que teste, como frequentemente ocorre |
| **R² — Teste** | **0,4967** | 🟡 Ajuste parcial | Melhor que uma previsão constante pela média |
| **R² médio CV** | **0,4447** | ✅ Positivo | Estimativa de validação relativamente próxima à do teste, mas menor |
| **Gap R² treino − teste** | **0,1833** | 🟡 Investigar | Indício de diferença de generalização, não prova isolada de overfitting |
| **RMSE — Baseline** | **73,22** | 🔴 Erro maior | Evidencia o valor adicional do XGBoost neste teste |

---

## 🧠 Dicas práticas para leitura

- **RMSE e MAE** devem ser lidos nas **unidades do `target`**, que neste dataset variou entre **25 e 346**.
- **RMSE é normalmente maior ou igual ao MAE**, porque amplifica a influência de erros grandes.
- **R² = 1** significa previsão perfeita para os valores avaliados; **R² = 0** corresponde ao erro quadrático da previsão constante pela média do conjunto avaliado; **R² negativo** é pior que essa referência.
- **R² = 0,4967 não significa 49,67% de acertos** nem que 49,67% dos pacientes foram corretamente diagnosticados. Estamos em **regressão**, não classificação.
- **R² CV** e **R² teste** são obtidos em protocolos diferentes e não precisam ser idênticos.
- **Treino muito melhor que validação** pode indicar *overfitting*. Para concluir, observe também os folds, erros, baseline e comportamento em novos dados.
- **Resíduos = real − predito**: positivos indicam subestimação; negativos, superestimação.
- **Correlação não é causalidade**; importância por ganho também não identifica causa clínica.
- A análise de `target` e correlação feita com o dataset completo é **descritiva**. Para decisões de seleção de variáveis e tuning, use apenas o treinamento para evitar vazamento de informações.
- O conjunto de dados é pequeno e não substitui validação clínica ou externa.

---

## 📘 Como Ler os Resultados

##### ✅ RMSE — Erros maiores recebem maior penalização

- **RMSE teste: 51,64.** A raiz do erro quadrático médio resume os desvios da previsão, dando peso maior aos erros mais extremos. O baseline alcançou **73,22**, mostrando uma redução relevante dos erros quadráticos em escala de RMSE.

##### ✅ MAE — Erro médio absoluto em unidades reais

- **MAE teste: 41,38.** Em média, as previsões ficaram **41,38 unidades do indicador** distantes dos valores verdadeiros, sem considerar o sinal do erro. Não significa que toda observação tenha erro de 41,38.

##### ✅ R² — Quanto o modelo explica no conjunto avaliado?

- **R² teste: 0,4967.** O XGBoost obteve redução de aproximadamente **49,67% na soma dos erros quadráticos** frente à previsão constante pela média dos valores reais do conjunto de teste. Isso não é uma taxa de acerto de pacientes.

##### 🟡 R² CV — Generalização estimada

- **R² médio CV: 0,4447** na execução completa. O resultado foi obtido durante a seleção de hiperparâmetros com cinco folds. O R² em treino é maior (**0,6800**); a diferença sugere que o modelo ajusta os dados conhecidos melhor do que os novos.

##### 🟡 Resíduos — O modelo erra para qual lado?

No projeto, o resíduo é calculado por:

```python
residuos = y_test - y_pred_test
```

- **Resíduo positivo:** valor real maior que a previsão → subestimativa.
- **Resíduo negativo:** valor real menor que a previsão → superestimativa.
- **Ideal:** ausência de tendência clara, distribuição razoavelmente centrada em zero e dispersão sem padrão sistemático.

**Atenção:** histograma simétrico, por si só, não garante ausência de viés em diferentes faixas do alvo.

---

## 🏆 Hiperparâmetros Selecionados na Execução Completa

| Hiperparâmetro | Resultado |
|---|---:|
| `learning_rate` | **0,03** |
| `n_estimators` | **300** |
| `reg_alpha` (L1) | **0,1** |
| `reg_lambda` (L2) | **10** |
| `max_depth` | **2 (fixo)** |
| `min_child_weight` | **5 (fixo)** |
| `subsample` | **0,8 (fixo)** |

O modelo da execução completa apresentou **R² médio CV de 0,4447**. O valor é muito próximo ao resultado anterior de 0,4430. A melhoria numérica **não demonstra, sozinha, superioridade consistente**.

---

## 📷 Visualizações Geradas

O código gera **sete gráficos**, preservando os quatro tipos principais do seu exemplo com AdaBoost e acrescentando verificações específicas para XGBoost.

| Gráfico | O que apresenta | Como interpretar |
|---|---|---|
| 🔵 **1. Dispersão Real vs Predito** | Previsões × valores reais, linha ideal e linha de tendência | Quanto mais próximos da linha ideal, menores os erros; distância indica desvio |
| 🟣 **2. Histograma dos Resíduos** | Distribuição de `real − predito` e média dos erros | Verificar centralização, assimetria e possíveis valores extremos |
| 🟠 **3. Comparação Ordenada Real vs Predito** | Séries ordenadas pelo valor real | Observar faixas nas quais o XGBoost subestima ou superestima sistematicamente |
| 📊 **4. Comparação das Métricas** | RMSE/MAE de treino, teste e baseline; R² em painel separado | Comparar erros na mesma escala e evitar misturar R² com RMSE em uma só régua |
| 🔍 **5. Resíduos vs Predito** | Erro residual no eixo vertical × previsão no horizontal | Detectar curvatura, variação desigual dos erros e padrões sistemáticos |
| 🌳 **6. Importância das Variáveis** | Importância relativa por ganho no XGBoost | Entender quais variáveis foram úteis para as divisões; **não implica causalidade** |
| 📉 **7. Variabilidade CV** | R² médio ± desvio-padrão das melhores configurações | Observar se aparentes melhorias são pequenas em relação à variação entre folds |

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import (
    train_test_split,
    KFold,
    GridSearchCV
)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
from xgboost import XGBRegressor


# ============================================================
# ⚙️ CONFIGURAÇÕES INICIAIS
# ============================================================

RANDOM_STATE = 42

PASTA = Path("resultados_xgboost")
PASTA.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")


# ============================================================
# 📊 FUNÇÕES AUXILIARES
# ============================================================

def metricas(y_real, y_previsto):
    """Calcula RMSE, MAE e R²."""

    return {
        "RMSE": float(
            np.sqrt(mean_squared_error(y_real, y_previsto))
        ),
        "MAE": float(
            mean_absolute_error(y_real, y_previsto)
        ),
        "R2": float(
            r2_score(y_real, y_previsto)
        )
    }


def salvar_figura(nome):
    """Salva o gráfico na pasta de resultados."""

    plt.tight_layout()

    plt.savefig(
        PASTA / nome,
        dpi=160,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"📊 Gráfico salvo: {PASTA / nome}",
        flush=True
    )


# ============================================================
# 📥 1. CARREGAMENTO DO DATASET
# ============================================================

print("\n" + "=" * 68)
print("⚡ PREDIÇÃO DO INDICADOR DE PROGRESSÃO DO DIABETES COM XGBOOST")
print("=" * 68)

dados = load_diabetes(as_frame=True)

X = dados.data
y = dados.target

df = dados.frame.copy()


# ============================================================
# 📊 2. ANÁLISE EXPLORATÓRIA DOS DADOS (EDA)
# ============================================================

print("\n📌 VISÃO GERAL DO DATASET:\n")
print(df.head())

print("\n📌 INFORMAÇÕES DO DATASET:\n")
df.info()

print("\n📌 DESCRIÇÃO ESTATÍSTICA:\n")
print(df.describe())

print("\n📌 VALORES AUSENTES:\n")
print(df.isnull().sum())

print("\n📌 CORRELAÇÃO COM O TARGET:\n")

correlacoes = (
    df.corr(numeric_only=True)["target"]
    .sort_values(ascending=False)
)

print(correlacoes)

print("\n📌 FORMATO DAS VARIÁVEIS:")
print(X.shape)

print("\n📌 INTERVALO DO TARGET:")
print(f"{y.min():.0f} a {y.max():.0f}")


# ============================================================
# 🔀 3. DIVISÃO TREINAMENTO E TESTE
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("\n🔀 DIVISÃO DOS DADOS:")

print(f"Treinamento: {len(X_train)} amostras")
print(f"Teste:       {len(X_test)} amostras")


# ============================================================
# 🔍 4. VALIDAÇÃO CRUZADA
# ============================================================

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 🤖 5. CONFIGURAÇÃO DO XGBOOST
# ============================================================

modelo_base = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    eval_metric="rmse",
    random_state=RANDOM_STATE,

    # Uma thread para evitar problemas de paralelismo
    n_jobs=1,

    verbosity=0,

    # Regularização
    reg_alpha=0.0,
    reg_lambda=1.0
)


# ============================================================
# ⚙️ 6. HIPERPARÂMETROS PARA GRIDSEARCHCV
# ============================================================

param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.03, 0.1],
    "max_depth": [2, 3, 4],
    "min_child_weight": [1, 5],
    "subsample": [0.8, 1.0]
}

# Total:
# 2 × 2 × 3 × 2 × 2 = 48 combinações
# 48 × 5 folds = 240 treinamentos
# + 1 treinamento final (refit)


# ============================================================
# 🏆 7. OTIMIZAÇÃO COM GRIDSEARCHCV
# ============================================================

grid = GridSearchCV(
    estimator=modelo_base,
    param_grid=param_grid,

    # Métrica utilizada para selecionar o melhor modelo
    scoring="r2",

    # Validação cruzada
    cv=cv,

    # CORREÇÃO PRINCIPAL:
    # Evitar paralelismo durante o diagnóstico
    n_jobs=1,

    # Exibir progresso do treinamento
    verbose=2,

    # Treinar novamente a melhor configuração
    refit=True,

    # Armazenar métricas de treino e validação
    return_train_score=True,

    # Mostrar erros em vez de ocultá-los
    error_score="raise"
)

print("\n" + "=" * 68)
print("🔍 INICIANDO BUSCA DE HIPERPARÂMETROS...")
print("=" * 68, flush=True)

grid.fit(X_train, y_train)

print("\n✅ GRIDSEARCH CONCLUÍDO!", flush=True)


# ============================================================
# 🥇 8. MELHOR MODELO
# ============================================================

melhor_modelo = grid.best_estimator_

melhor_idx = grid.best_index_

print("\n" + "=" * 68)
print("🏆 MELHOR COMBINAÇÃO DE HIPERPARÂMETROS")
print("=" * 68)

for parametro, valor in grid.best_params_.items():
    print(f"{parametro}: {valor}")

print("\n📊 MELHOR R² MÉDIO NA VALIDAÇÃO CRUZADA:")
print(f"{grid.best_score_:.4f}")

print("\n✅ Modelo otimizado e treinado com sucesso!")



⚡ PREDIÇÃO DO INDICADOR DE PROGRESSÃO DO DIABETES COM XGBOOST

📌 VISÃO GERAL DO DATASET:

        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  target  
0 -0.002592  0.019907 -0.017646   151.0  
1 -0.039493 -0.068332 -0.092204    75.0  
2 -0.002592  0.002861 -0.025930   141.0  
3  0.034309  0.022688 -0.009362   206.0  
4 -0.002592 -0.031988 -0.046641   135.0  

📌 INFORMAÇÕES DO DATASET:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  